# Практика · Оцінка й упередженість ембедингів

> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.html](homework.html) ·
> Тест: [quiz.html](quiz.html)

Тут рахуються **всі** числа, які називає лекція, і в тому самому порядку.

Що зробимо:

1. Зберемо корпус і навчимо **три** моделі skip-gram — по одній на зерно.
2. Поставимо внутрішню оцінку: пари документів гарантовано однакового змісту.
3. Порівняємо ембединги з TF-IDF **трьома** мірами й дістанемо три різні відповіді.
4. Знайдемо спільний напрямок, який завищує геть усі косинуси, і приберемо його.
5. Заміряємо, як шкала косинуса пливе разом із частотою слова, і перевіримо словоформи.
6. Поставимо **зовнішню** оцінку: візьмемо задачу теми 06 і подивимось, чи допомагають ембединги.
7. Заміряємо упередженість: домен, сусідство, гендер і те, чого в корпусі немає взагалі.

> ⏱ Зошит навчає три моделі skip-gram з нуля. Заміряно: близько **чотирьох хвилин**
> на чотирьох ядрах без відеокарти. Найдовше йдуть три навчання (по 25-28 секунд
> процесорного часу кожне) і крива від кількості розмічених прикладів.

## 1 · Середовище

Перша клітинка друкує версії й просить числові бібліотеки рахувати **в один потік**.

Це не оптимізація, а умова того, щоб замір часу щось означав. На машині, де вже щось
рахується, чотири потоки більшу частину часу чекають одне на одного — і це очікування
записується в процесорний час як робота. Ставити змінні треба **до** імпорту numpy:
пізніше вони вже не подіють.

In [ ]:
import os
# Просимо numpy, scipy й torch рахувати в один потік.
# Рядки мусять стояти ДО імпорту numpy — інакше бібліотека вже прочитала своє.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import sys, re, math, glob, gettext, time, collections
import numpy as np
import sklearn
import torch
torch.set_num_threads(1)

print("Python  ", sys.version.split()[0])
print("numpy   ", np.__version__)
print("sklearn ", sklearn.__version__)
print("torch   ", torch.__version__)
print("потоків :", os.environ["OMP_NUM_THREADS"])

## 2 · Корпус: ті самі переклади інтерфейсів

Корпус той самий, що в усьому курсі: пари «англійський оригінал → український
переклад» із `.mo`-файлів української локалі.

Якщо української локалі на машині немає, вмикається маленький вбудований запасний
корпус. Він друкує гучне попередження: числа на ньому будуть **інші**, і посилатися
на них не можна.

In [ ]:
def load_system_corpus():
    """Читаємо .mo-файли української локалі.
    Повертаємо трійки (програма, англійський оригінал, український переклад)."""
    docs = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                       # зламаний або чужий формат — пропускаємо
        program = path.split('/')[-1][:-3]
        for source, target in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і не є текстом
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > 30 and 'Project-Id' not in target:
                docs.append((program, source, target))
    return docs


FALLBACK = [
    ("fallback", "Could not open the configuration file",
     "Не вдалося відкрити файл налаштувань програми"),
    ("fallback", "Invalid certificate chain for the remote server",
     "Некоректний ланцюжок сертифікатів віддаленого сервера"),
    ("fallback", "The directory tree is too deep to display",
     "Дерево каталогів надто глибоке, щоб його показати"),
    ("fallback", "Failed to write to the output stream",
     "Не вдалося записати дані у потік виведення"),
    ("fallback", "Unable to read the input file: permission denied",
     "Неможливо прочитати вхідний файл: доступ заборонено"),
    ("fallback", "Show the dialog window on the primary monitor",
     "Показувати діалогове вікно на основному моніторі"),
]

corpus = load_system_corpus()
if len(corpus) < 5000:
    print("⚠️  УВАГА: української локалі на цій машині немає або вона надто мала.")
    print("⚠️  Вмикається запасний мінікорпус. Усі числа нижче будуть ІНШИМИ,")
    print("⚠️  і посилатися на них не можна.")
    corpus = FALLBACK * 900
print("документів :", len(corpus))
print("програм    :", len(set(program for program, _, _ in corpus)))
print("приклад    :", corpus[0][2][:70])

## 3 · Токенізатор, словник і те, чого модель не побачить

Токенізатор — канон курсу: букви української абетки, апостроф всередині слова
не розриває його на два токени.

`min_count=10` — поріг, нижче за який слово в модель не потрапляє взагалі. Порахуємо
одразу, скільки словоформ цей поріг викидає й скільки тексту вони важать. Ці два
числа — не те саме, і різниця між ними і є хвостом Ципфа з теми 01.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"     # канон курсу, тема 04
_token_re = re.compile(TOKEN_PATTERN)

def tokenize(text):
    """Малі літери й тільки українські слова. Апостроф — звʼязка, а не межа."""
    return _token_re.findall(text.lower())

programs  = [program for program, _, _ in corpus]
sources   = [source  for _, source, _ in corpus]     # англійський бік: мітка
documents = [target  for _, _, target in corpus]     # український бік: ознаки
tokenized = [tokenize(text) for text in documents]

frequency = collections.Counter()
for words in tokenized:
    frequency.update(words)

MIN_COUNT = 10
vocabulary = [word for word, count in frequency.most_common() if count >= MIN_COUNT]
word_index = {word: i for i, word in enumerate(vocabulary)}
word_counts = np.array([frequency[word] for word in vocabulary])

total_tokens = sum(frequency.values())
covered = int(word_counts.sum())
rare_types = len(frequency) - len(vocabulary)
rare_tokens = total_tokens - covered

print("слововживань          :", total_tokens)
print("різних словоформ      :", len(frequency))
print("словник (count >= 10) :", len(vocabulary))
print(f"покриття тексту       : {100 * covered / total_tokens:.4f} %")
print()
print(f"поза моделлю: {rare_types} словоформ = "
      f"{100 * rare_types / len(frequency):.2f} % словника, "
      f"але лише {rare_tokens} слововживань = {100 * rare_tokens / total_tokens:.2f} % тексту")

## 4 · Skip-gram своїми руками

`gensim` на цій машині немає, тож пишемо самі — і це на краще: формула має бути
видима.

Одна ідея в двох рядках. Беремо слово й одного з його сусідів у вікні ±5 і просимо
модель сказати, що ця пара **справжня**. Потім беремо те саме слово й пʼять
випадкових слів і просимо сказати, що ці пари **вигадані**. Усе, чого модель
навчиться, — це відрізняти справжніх сусідів від випадкових; вектори виходять
побічним продуктом.

Одна деталь, без якої нічого не працює: **проріджування частих слів**. Слово «не»
трапляється в кожному пʼятнадцятому документі, і без проріджування воно з'їдає
навчання. Формула Word2Vec лишає слово з імовірністю, що падає з його частотою.

In [ ]:
def build_pairs(sequences, counts, total, window=5, threshold=1e-3, rng=None):
    """Пари «слово — сусід» у вікні ±window, у межах одного документа.

    threshold вмикає проріджування частих слів: слово з часткою f лишається
    з імовірністю (sqrt(f/t) + 1) * t/f. Для рідкісних слів це одиниця,
    для «не» — близько сотої."""
    if threshold:
        share = counts / total
        keep = np.minimum(1.0, (np.sqrt(share / threshold) + 1) * (threshold / share))
    else:
        keep = np.ones(len(counts))          # без проріджування: лишаємо все
    lengths = np.array([len(s) for s in sequences], dtype=np.int64)
    flat = np.concatenate([s for s in sequences if len(s) > 0]).astype(np.int64)
    doc_id = np.repeat(np.arange(len(sequences)), lengths)
    alive = rng.random(len(flat)) < keep[flat]      # кидаємо монету на кожне слово
    flat, doc_id = flat[alive], doc_id[alive]

    left, right = [], []
    for offset in range(1, window + 1):
        same_doc = doc_id[:-offset] == doc_id[offset:]
        a = flat[:-offset][same_doc]
        b = flat[offset:][same_doc]
        left.append(a); right.append(b)            # сусід праворуч
        left.append(b); right.append(a)            # і той самий сусід ліворуч
    return np.concatenate(left), np.concatenate(right)


def train_skipgram(centers, contexts, vocab_size, counts,
                   dim=64, negative=5, epochs=3, lr=0.025, seed=0, batch=8192):
    """Skip-gram із негативним семплюванням. Два набори векторів:
    Win — центральна матриця, Wout — контекстна. Ембедингами зазвичай називають
    Win, але це домовленість, а не закон: повертаємо обидві, бо далі порівняємо."""
    gen = torch.Generator().manual_seed(seed)
    Win = (torch.rand(vocab_size, dim, generator=gen) - 0.5) / dim
    Wout = torch.zeros(vocab_size, dim)
    # частоти в степені 0.75: рідкісні слова трапляються негативами частіше,
    # ніж пропорційно своїй частоті — так радить стаття Word2Vec
    probs = torch.tensor(counts, dtype=torch.float64) ** 0.75
    probs = (probs / probs.sum()).float()

    n = len(centers)
    C = torch.from_numpy(centers); O = torch.from_numpy(contexts)
    step, total_steps = 0, epochs * ((n + batch - 1) // batch)
    for _ in range(epochs):
        order = torch.randperm(n, generator=gen)
        for start in range(0, n, batch):
            pick = order[start:start + batch]
            center, context = C[pick], O[pick]
            k = center.shape[0]
            fake = torch.multinomial(probs, k * negative, replacement=True,
                                     generator=gen).view(k, negative)
            v_in   = Win[center]                              # (k, dim)
            v_true = Wout[context]                            # (k, dim)
            v_fake = Wout[fake]                               # (k, negative, dim)
            # сигмоїда від скалярного добутку — наскільки модель вірить у пару
            p_true = torch.sigmoid((v_in * v_true).sum(1))
            p_fake = torch.sigmoid(torch.einsum('kd,knd->kn', v_in, v_fake))
            g_true = (p_true - 1.0).unsqueeze(1)              # хочемо одиницю
            g_fake = p_fake                                   # хочемо нуль
            grad_in = g_true * v_true + torch.einsum('kn,knd->kd', g_fake, v_fake)
            cur_lr = lr * max(1e-4, 1.0 - step / total_steps) # крок згасає лінійно
            Win.index_add_(0, center, -cur_lr * grad_in)
            Wout.index_add_(0, context, -cur_lr * (g_true * v_in))
            Wout.index_add_(0, fake.reshape(-1),
                            -cur_lr * (g_fake.unsqueeze(2) * v_in.unsqueeze(1)).reshape(-1, dim))
            step += 1
    return Win.numpy(), Wout.numpy()


# Клітинка не рахує нічого, крім однієї речі, яку варто побачити очима:
# наскільки жорстко проріджування ріже найчастіші слова.
share_of_text = word_counts / total_tokens
keep_probability = np.minimum(1.0, (np.sqrt(share_of_text / 1e-3) + 1) * (1e-3 / share_of_text))
print("проріджування частих слів (threshold = 1e-3):")
for word in ('не', 'для', 'файл', 'помилка', 'сертифікат'):
    i = word_index[word]
    print(f"   {word:<12} трапляється {word_counts[i]:>6} разів"
          f" · лишиться {100 * keep_probability[i]:6.2f} % входжень")

### Навчаємо три моделі — по одній на зерно

Одна модель нічого не доводить: зерно міняє і проріджування, і початкові вектори, і
порядок прикладів. Далі кожне число ми друкуватимемо разом із розкидом по трьох
зернах, і різницю, меншу за розкид, різницею не вважатимемо.

Час міряємо `time.process_time()` — процесорним годинником. Стінний на спільній
машині бреше.

In [ ]:
SEEDS = (0, 1, 2)

sequences = [np.array([word_index[w] for w in words if w in word_index], dtype=np.int64)
             for words in tokenized]

# скільки пар дає вікно ±5 БЕЗ проріджування — це розмір задачі
raw_centers, raw_contexts = build_pairs(sequences, word_counts, total_tokens,
                                        threshold=None,
                                        rng=np.random.default_rng(0))
print("пар «слово — сусід» при вікні ±5 :", len(raw_centers))

embeddings, context_vectors = {}, {}
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    centers, contexts = build_pairs(sequences, word_counts, total_tokens, rng=rng)
    started = time.process_time()
    embeddings[seed], context_vectors[seed] = train_skipgram(
        centers, contexts, len(vocabulary), word_counts, seed=seed)
    spent = time.process_time() - started
    print(f"зерно {seed}: пар після проріджування {len(centers)}, "
          f"навчання {spent:.1f} с процесорного часу")

Перевіримо очима, що вектори взагалі щось вивчили. Косинусна близькість — це
скалярний добуток векторів, поділених на свою довжину; поділимо один раз наперед,
щоб далі скрізь був просто добуток.

In [ ]:
def normalize_rows(matrix):
    """Ділимо кожен рядок на його довжину. Після цього скалярний добуток
    двох рядків і є косинусом кута між ними."""
    length = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(length, 1e-12)

unit = {seed: normalize_rows(embeddings[seed]) for seed in SEEDS}
# та сама модель, інший спосіб зібрати вектор слова: центральна плюс контекстна
unit_context = {seed: normalize_rows(context_vectors[seed]) for seed in SEEDS}
unit_sum = {seed: normalize_rows(embeddings[seed] + context_vectors[seed]) for seed in SEEDS}

# перевірка: наш косинус = бібліотечний
from sklearn.metrics.pairwise import cosine_similarity
probe = [word_index[w] for w in ('файл', 'помилка', 'вікно', 'каталог')]
ours = unit[0][probe] @ unit[0][probe].T
theirs = cosine_similarity(embeddings[0][probe])
assert np.allclose(ours, theirs, atol=1e-6), "розрахунок розійшовся!"
print("✅ наш косинус збігається з sklearn")
print()

def neighbours(seed, word, k=5):
    scores = unit[seed] @ unit[seed][word_index[word]]
    scores[word_index[word]] = -9                 # саме себе не показуємо
    return [vocabulary[i] for i in np.argsort(-scores)[:k]]

for word in ('файл', 'помилка', 'вікно', 'мережі'):
    print(f"{word:10} → " + " · ".join(neighbours(0, word)))

## 5 · Внутрішня оцінка: пари, про які точно відомо, що вони про одне

Готових українських наборів «наскільки схожі ці два слова» не існує, тож набір
доводиться робити самим — а набір, зроблений із того самого корпусу, на якому вчилася
модель, робить оцінку круговою.

Один чесний вихід у нас є, і його вже знайшла тема 05. Той самий англійський рядок
різні команди переклали по-різному, тож ми маємо пари документів, про які **точно**
відомо, що вони означають одне й те саме — і які при цьому написані різними словами.

In [ ]:
NOISE = re.compile(r"[@<>]|https?://")

def paraphrase_pairs():
    """Пари українських перекладів того самого англійського рядка,
    у яких майже немає спільних слів. Спосіб теми 05, дослівно."""
    by_source = collections.defaultdict(dict)
    for source, target in zip(sources, documents):
        if source.strip() == 'translator-credits':
            continue
        by_source[source][' '.join(target.split())] = 1
    found = []
    for source, variants in by_source.items():
        texts = list(variants)
        for i in range(len(texts)):
            for j in range(i + 1, len(texts)):
                first, second = texts[i], texts[j]
                if NOISE.search(first) or NOISE.search(second):
                    continue
                a = set(tokenize(first)); b = set(tokenize(second))
                if len(a) < 3 or len(b) < 3:
                    continue
                if len(a & b) / len(a | b) < 0.34:      # майже немає спільних слів
                    found.append((first, second, len(a & b)))
    return found

pairs = paraphrase_pairs()
print(f"пар «те саме іншими словами»: {len(pairs)}, "
      f"з них без жодного спільного слова: {sum(1 for p in pairs if p[2] == 0)}")
print()
for first, second, shared in pairs[:3]:
    print(f"  спільних слів {shared}")
    print(f"    A: {first}")
    print(f"    B: {second}")

### Документ як середнє своїх слів

Ембединги дають вектор слову, а порівнюємо ми документи. Найпростіший міст —
середнє векторів усіх слів документа. Він грубий (порядок слів зникає), зате
не має жодного налаштування, яке можна було б підкрутити під потрібну відповідь.

І одразу домовимось про чесність порівняння. Тема 05 міряла TF-IDF на вибірці
**20 000 документів**, по одній на зерно, і брала контрольні випадкові пари з тієї
самої вибірки. Ми повторюємо її вибірку **дослівно** — тим самим генератором і в
тому самому порядку, — щоб наші числа TF-IDF збіглися з її числами символ у символ,
а ембединги міряти на **тих самих** парах.

In [ ]:
N_SAMPLE = 20000          # рівно стільки документів брала тема 05

def sample_documents(seed, n=N_SAMPLE):
    """Ті самі 20 000 документів при тому самому зерні — спосіб теми 05, дослівно."""
    rng = np.random.default_rng(seed)
    picked = rng.choice(len(documents), min(n, len(documents)), replace=False)
    return [documents[i] for i in picked]


def control_indices(texts, seed):
    """Індекси контрольних випадкових пар. Порядок звертань до генератора
    той самий, що в темі 05, — інакше пари вийдуть інші."""
    rng = np.random.default_rng(7 + seed)
    left = rng.choice(len(texts), len(pairs))
    right = rng.choice(len(texts), len(pairs))
    return left, right


def document_vector(text, matrix):
    """Середнє векторів слів документа, зведене до одиничної довжини.
    Слова поза словником просто пропускаємо — вектора в них немає."""
    ids = [word_index[w] for w in tokenize(text) if w in word_index]
    if not ids:
        return None
    vector = matrix[ids].mean(0)
    length = np.linalg.norm(vector)
    return vector / length if length > 0 else None


def cosines_of(first_texts, second_texts, matrix):
    """Косинуси між двома списками документів. Пари, де хоч один документ
    не має жодного відомого слова, пропускаємо."""
    out = []
    for first, second in zip(first_texts, second_texts):
        a = document_vector(first, matrix)
        b = document_vector(second, matrix)
        if a is not None and b is not None:
            out.append(float(a @ b))
    return np.array(out)


def similarity_sets(matrix, seed):
    """Дві купи чисел: косинуси пар однакового змісту й косинуси
    випадкових пар документів тієї самої вибірки."""
    texts = sample_texts[seed]
    left, right = control_indices(texts, seed)
    same = cosines_of([p[0] for p in pairs], [p[1] for p in pairs], matrix)
    other = cosines_of([texts[i] for i in left], [texts[i] for i in right], matrix)
    return same, other


sample_texts = {seed: sample_documents(seed) for seed in SEEDS}

emb_same, emb_other = {}, {}
for seed in SEEDS:
    emb_same[seed], emb_other[seed] = similarity_sets(unit[seed], seed)
    print(f"зерно {seed}: однаковий зміст {emb_same[seed].mean():.4f}"
          f" · випадкові пари {emb_other[seed].mean():.4f}"
          f" · розрив {emb_same[seed].mean() - emb_other[seed].mean():.4f}")

### Те саме на TF-IDF — щоб було з чим порівняти

Ваги з теми 05: той самий токенізатор, та сама вибірка, ті самі 209 пар і ті самі
контрольні пари. Якщо ми все зробили правильно, числа мають збігтися з тими, що
надрукувала тема 05.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_same, tfidf_other = {}, {}
for seed in SEEDS:
    texts = sample_texts[seed]
    tfidf = TfidfVectorizer(token_pattern=TOKEN_PATTERN)      # налаштування теми 05
    tfidf.fit(texts)
    L = tfidf.transform([p[0] for p in pairs])
    R = tfidf.transform([p[1] for p in pairs])
    tfidf_same[seed] = np.asarray(L.multiply(R).sum(axis=1)).ravel()
    left, right = control_indices(texts, seed)
    A = tfidf.transform([texts[i] for i in left])
    B = tfidf.transform([texts[i] for i in right])
    tfidf_other[seed] = np.asarray(A.multiply(B).sum(axis=1)).ravel()
    print(f"зерно {seed}: однаковий зміст {tfidf_same[seed].mean():.4f}"
          f" · випадкові пари {tfidf_other[seed].mean():.4f}"
          f" · розрив {tfidf_same[seed].mean() - tfidf_other[seed].mean():.4f}")

# Тема 05 надрукувала для тих самих зерен 0.2230 / 0.2165 / 0.2200.
# Якщо ми відтворили її вибірку, наші числа мусять збігтися до четвертого знака.
published = [0.2230, 0.2165, 0.2200]
ours = [round(float(tfidf_same[seed].mean()), 4) for seed in SEEDS]
assert ours == published, f"розійшлися з темою 05: {ours} проти {published}"
print()
print("✅ наші числа TF-IDF збіглися з темою 05:", ours)
print()

# Скільки пар однакового змісту дістає рівно нуль — і скільки їх опиняється
# нижче за середню випадкову пару. Друге питання має сенс для обох систем,
# перше — лише для розрідженої.
for name, same_of, other_of in (("TF-IDF", tfidf_same, tfidf_other),
                                ("ембединги", emb_same, emb_other)):
    zeros = np.mean([np.mean(same_of[s] == 0) for s in SEEDS])
    below = np.mean([np.mean(same_of[s] < other_of[s].mean()) for s in SEEDS])
    print(f"{name:<11} рівно нуль у {100 * zeros:5.1f} % пар однакового змісту"
          f" · нижче за середню випадкову пару — {100 * below:5.1f} %")

## 6 · Три міри, три відповіді

Тепер зведемо. У нас дві системи й ті самі дві купи чисел. Порівняємо їх трьома
способами й подивимось, чи дадуть вони одну відповідь.

**Спосіб 1 — сирий косинус на парах однакового змісту.** Так цитують найчастіше.

**Спосіб 2 — розрив** між парами однакового змісту й випадковими парами. Він знімає
загальний перекіс шкали: якщо система завищує всі косинуси однаково, розрив цього
не помітить.

**Спосіб 3 — AUC**: імовірність того, що навмання взята пара однакового змісту дістане
вищу оцінку, ніж навмання взята випадкова пара. Це число взагалі не залежить від
шкали — його не змінить жодне зростаюче перетворення осі.

In [ ]:
from sklearn.metrics import roc_auc_score

def separation_auc(same, other):
    """Наскільки добре міра відрізняє «те саме» від «випадкове».
    0.5 — не відрізняє зовсім, 1.0 — жодного переплутування."""
    y = np.r_[np.ones(len(same)), np.zeros(len(other))]
    return roc_auc_score(y, np.r_[same, other])


def three_measures(same_of, other_of):
    """Сирий косинус, розрив і AUC — кожне число по трьох зернах."""
    raw = np.array([same_of[s].mean() for s in SEEDS])
    gap = np.array([same_of[s].mean() - other_of[s].mean() for s in SEEDS])
    auc = np.array([separation_auc(same_of[s], other_of[s]) for s in SEEDS])
    return raw, gap, auc


def proven(diff, spread):
    """Різниця, менша за подвоєний розкид, різницею не вважається."""
    return "різниця доведена" if abs(diff) > 2 * spread else "у межах розкиду — не різниця"


raw_t, gap_t, auc_t = three_measures(tfidf_same, tfidf_other)
raw_e, gap_e, auc_e = three_measures(emb_same, emb_other)

rnd_t = np.array([tfidf_other[s].mean() for s in SEEDS])
rnd_e = np.array([emb_other[s].mean() for s in SEEDS])

print(f"{'система':<12}{'сирий косинус':>22}{'випадкові пари':>22}{'розрив':>20}{'AUC':>20}")
for name, raw, rnd, gap, auc in (("TF-IDF", raw_t, rnd_t, gap_t, auc_t),
                                 ("ембединги", raw_e, rnd_e, gap_e, auc_e)):
    print(f"{name:<12}{raw.mean():>14.4f} ±{raw.std():.4f}"
          f"{rnd.mean():>14.4f} ±{rnd.std():.4f}"
          f"{gap.mean():>12.4f} ±{gap.std():.4f}{auc.mean():>12.4f} ±{auc.std():.4f}")
print()
print(f"сирий косинус: ембединги більші у {raw_e.mean() / raw_t.mean():.2f} раза")
print(f"розрив       : {gap_e.mean() - gap_t.mean():+.4f} при розкиді "
      f"{max(gap_e.std(), gap_t.std()):.4f} → "
      f"{proven(gap_e.mean() - gap_t.mean(), max(gap_e.std(), gap_t.std()))}")
print(f"AUC          : {auc_e.mean() - auc_t.mean():+.4f} при розкиді "
      f"{max(auc_e.std(), auc_t.std()):.4f} → "
      f"{proven(auc_e.mean() - auc_t.mean(), max(auc_e.std(), auc_t.std()))}")

## 7 · Той самий простір, три різні відповіді

Три міри дали три різні вироки на одній парі систем. Тепер зробимо гірше: не
змінюючи ані моделі, ані даних, ані набору пар, дістанемо **три різні оцінки самих
ембедингів**. Достатньо по-різному зібрати вектор слова з того, що модель уже
навчила.

Skip-gram тримає **дві** матриці. `Win` — центральна: рядок для слова, коли воно
стоїть у центрі вікна. `Wout` — контекстна: рядок для того самого слова, коли воно
стоїть сусідом. Ембедингами зазвичай беруть `Win` — але це домовленість, а не
закон, і сума `Win + Wout` теж цілком законна.

Порахуємо для кожного способу дві речі: середній косинус між **усіма** парами слів
словника (наскільки простір узагалі «злиплий») і три наші мірки на 209 парах.

In [ ]:
def mean_pairwise_cosine(matrix):
    """Середній косинус між усіма різними парами слів словника.
    Рахуємо не перебором мільйонів пар, а через суму векторів: сума всіх попарних
    добутків дорівнює квадрату довжини суми, з неї віднімаємо n одиниць діагоналі."""
    rows = normalize_rows(matrix)
    n = rows.shape[0]
    total = float(np.dot(rows.sum(0), rows.sum(0)))
    return (total - n) / (n * (n - 1))


VARIANTS = [
    ("центральна Win (як досі)", unit),
    ("контекстна Wout", unit_context),
    ("сума Win + Wout", unit_sum),
]

print(f"{'спосіб зібрати вектор':<28}{'сер. косинус усіх пар':>23}"
      f"{'однаковий зміст':>17}{'випадкові':>12}{'розрив':>19}{'AUC':>19}")
variant_rows = {}
for name, matrices in VARIANTS:
    glue = float(np.mean([mean_pairwise_cosine(matrices[s]) for s in SEEDS]))
    same_of, other_of = {}, {}
    for seed in SEEDS:
        same_of[seed], other_of[seed] = similarity_sets(matrices[seed], seed)
    raw, gap, auc = three_measures(same_of, other_of)
    other = np.array([other_of[s].mean() for s in SEEDS])
    variant_rows[name] = (glue, raw, other, gap, auc, same_of, other_of)
    print(f"{name:<28}{glue:>23.4f}{raw.mean():>17.4f}{other.mean():>12.4f}"
          f"{gap.mean():>13.4f} ±{gap.std():.4f}{auc.mean():>13.4f} ±{auc.std():.4f}")

gap_win = variant_rows["центральна Win (як досі)"][3]
gap_sum = variant_rows["сума Win + Wout"][3]
print()
print(f"розрив: центральна {gap_win.mean():.4f} ±{gap_win.std():.4f}"
      f" · сума {gap_sum.mean():.4f} ±{gap_sum.std():.4f}"
      f" · різниця {gap_sum.mean() - gap_win.mean():+.4f}")
print(f"для порівняння, вся різниця між TF-IDF і центральною матрицею: "
      f"{gap_win.mean() - gap_t.mean():+.4f}")
print()
print("Тобто одна деталь реалізації зсуває оцінку сильніше, ніж уся заміна методу.")

### Чому сума працює краще: спільний напрямок

Причина видна в першому стовпчику. У центральній матриці **середній косинус між
будь-якими двома словами словника** високий: вектори дивляться приблизно в один бік.
Такий простір називають **анізотропним** (anisotropic) — він розтягнутий уздовж
одного напрямку сильніше, ніж уздовж решти.

Звідки цей напрямок береться. Skip-gram весь час робить дві речі: тягне слово до
справжніх сусідів і відштовхує від пʼятьох випадкових «негативів». Відштовхування
діє на всі слова майже однаково, бо негативи беруться з усього словника, — і за два
з половиною мільйони кроків спільний штовхан накопичується. У контекстній матриці
він накопичується в **інший** бік, тому сума їх частково гасить.

Перевірити здогад можна прямо, без жодної другої матриці: відняти від кожного
вектора частку `alpha` середнього вектора словника й подивитись, що станеться.

In [ ]:
def centered_matrix(matrix, alpha):
    """Прибираємо частку alpha спільного напрямку: у кожного вектора віднімаємо
    alpha × середній вектор словника й знову зводимо до одиничної довжини."""
    common = matrix.mean(0)
    return normalize_rows(matrix - alpha * common)


common_length = np.mean([np.linalg.norm(unit[s].mean(0)) for s in SEEDS])
print(f"довжина середнього вектора центральної матриці: {common_length:.4f}")
print("(одиниця означала б, що всі вектори дивляться строго в один бік,")
print(" нуль — що спільного напрямку немає взагалі)")
print()

ALPHAS = [0.0, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50]
sweep = {}
print(f"{'alpha':>7}{'однаковий зміст':>18}{'випадкові':>13}{'розрив':>18}{'AUC':>18}")
for alpha in ALPHAS:
    same_of, other_of = {}, {}
    for seed in SEEDS:
        same_of[seed], other_of[seed] = similarity_sets(centered_matrix(unit[seed], alpha), seed)
    raw, gap, auc = three_measures(same_of, other_of)
    other = np.array([other_of[s].mean() for s in SEEDS])
    sweep[alpha] = (raw, other, gap, auc)
    print(f"{alpha:>7.2f}{raw.mean():>13.4f}{other.mean():>13.4f}"
          f"{gap.mean():>12.4f} ±{gap.std():.4f}{auc.mean():>12.4f} ±{auc.std():.4f}")

best_alpha = max(ALPHAS, key=lambda a: sweep[a][2].mean())
raw_c, other_c, gap_c, auc_c = sweep[best_alpha]
print()
print(f"найбільший розрив дає alpha = {best_alpha:.2f}")
print(f"розрив при ньому: {gap_c.mean():.4f} ±{gap_c.std():.4f}"
      f"  проти {gap_e.mean():.4f} без центрування,"
      f"  {gap_sum.mean():.4f} у суми Win + Wout"
      f"  і {gap_t.mean():.4f} у TF-IDF")
print(f"AUC при ньому {auc_c.mean():.4f} проти {auc_e.mean():.4f} — "
      f"{proven(auc_c.mean() - auc_e.mean(), max(auc_c.std(), auc_e.std()))}")

Тепер видно, звідки бралися різні відповіді. Сирий косинус міряв **спільну
складову плюс зміст**, розрив — те, що лишається після віднімання спільного рівня,
а AUC на центрування майже не зреагував, бо порядок пар від нього змінюється слабо.

Останнє й найважливіше: розподіли цілком. Розкладемо купи чисел по кошиках — саме ці
числа малює перша фігура лекції.

In [ ]:
BUCKETS_FROM, BUCKETS_TO, BUCKETS_N = -0.2, 1.0, 24
edges = np.linspace(BUCKETS_FROM, BUCKETS_TO, BUCKETS_N + 1)

centered_same, centered_other = {}, {}
for seed in SEEDS:
    centered_same[seed], centered_other[seed] = similarity_sets(
        centered_matrix(unit[seed], best_alpha), seed)

sum_same = variant_rows["сума Win + Wout"][5]
sum_other = variant_rows["сума Win + Wout"][6]

REPRESENTATIONS = [
    ("TF-IDF", tfidf_same, tfidf_other),
    ("ембединги, центральна Win", emb_same, emb_other),
    ("ембединги, сума Win + Wout", sum_same, sum_other),
    (f"ембединги, спільний напрямок прибрано (alpha {best_alpha:.2f})",
     centered_same, centered_other),
]

for name, same_of, other_of in REPRESENTATIONS:
    hist_same = np.histogram(same_of[0], bins=edges)[0]
    hist_other = np.histogram(other_of[0], bins=edges)[0]
    print(f"{name} (зерно 0, {len(same_of[0])} пар у кожній купі)")
    print("   однаковий зміст :", hist_same.tolist())
    print("   випадкові пари  :", hist_other.tolist())
print()
print(f"кошики від {BUCKETS_FROM} до {BUCKETS_TO}, ширина "
      f"{(BUCKETS_TO - BUCKETS_FROM) / BUCKETS_N:.2f}")
print()
print(f"{'система':<52}{'сирий':>9}{'розрив':>9}{'AUC':>9}")
for name, same_of, other_of in REPRESENTATIONS:
    raw, gap, auc = three_measures(same_of, other_of)
    print(f"{name:<52}{raw.mean():>9.4f}{gap.mean():>9.4f}{auc.mean():>9.4f}")

## 8 · Чому сирий косинус не має єдиної шкали

Відповідь видно, якщо взяти **випадкові** пари слів і подивитись, як їхній косинус
залежить від того, наскільки часті ці слова. Ділимо словник на чотири смуги за
рангом частоти й міряємо в кожній.

In [ ]:
rank = np.argsort(np.argsort(-word_counts))          # 0 — найчастіше слово
BANDS = [(0, 300), (300, 1000), (1000, 3000), (3000, len(vocabulary))]

print(f"{'смуга рангів':>16}{'частота':>16}{'косинус випадкової пари':>26}")
band_random = {}
for low, high in BANDS:
    band = np.where((rank >= low) & (rank < high))[0]
    values = []
    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        a = rng.choice(band, 20000); b = rng.choice(band, 20000)
        different = a != b
        values.append(float((unit[seed][a[different]] * unit[seed][b[different]]).sum(1).mean()))
    band_random[(low, high)] = (np.mean(values), np.std(values))
    print(f"{f'{low}-{high}':>16}{f'{word_counts[band].min()}-{word_counts[band].max()}':>16}"
          f"{np.mean(values):>18.4f} ±{np.std(values):.4f}")

І другий бік того самого: у рідкісного слова сусіди щоразу інші. Порахуємо, яка
частка з пʼятьох найближчих сусідів збігається між двома зернами.

In [ ]:
def top_neighbours(matrix, k=5):
    """Для кожного слова — індекси k найближчих. argpartition замість
    повного сортування: нам потрібні лише перші k, а не порядок усіх."""
    scores = matrix @ matrix.T
    np.fill_diagonal(scores, -9)
    top = np.argpartition(-scores, k, axis=1)[:, :k]
    return top

tops = {seed: top_neighbours(unit[seed]) for seed in SEEDS}

print(f"{'смуга рангів':>16}{'частота':>16}{'спільних сусідів':>20}")
band_stability = {}
for low, high in BANDS:
    band = np.where((rank >= low) & (rank < high))[0]
    shares = []
    for i in band:
        a, b, c = set(tops[0][i]), set(tops[1][i]), set(tops[2][i])
        shares.append((len(a & b) + len(a & c) + len(b & c)) / 15)
    band_stability[(low, high)] = float(np.mean(shares))
    print(f"{f'{low}-{high}':>16}{f'{word_counts[band].min()}-{word_counts[band].max()}':>16}"
          f"{np.mean(shares):>20.4f}")

## 9 · Друга внутрішня перевірка: словоформи однієї леми

Українська дає нам ще один набір пар, який не треба вигадувати. `pymorphy3` зводить
словоформу до леми, і всі форми однієї леми — гарантовано про одне й те саме слово.

Це перевірка не на зміст, а на морфологію: чи стоять «файл» і «файлу» поруч.

In [ ]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer(lang='uk')

by_lemma = collections.defaultdict(list)
for word in vocabulary:
    parsed = morph.parse(word)
    by_lemma[parsed[0].normal_form if parsed else word].append(word)

form_pairs = []
for lemma, forms in by_lemma.items():
    for i in range(len(forms)):
        for j in range(i + 1, len(forms)):
            form_pairs.append((word_index[forms[i]], word_index[forms[j]]))
form_pairs = np.array(form_pairs)

print(f"лем серед {len(vocabulary)} словоформ словника: {len(by_lemma)}")
print(f"пар «різні форми однієї леми»: {len(form_pairs)}")
print()
rng = np.random.default_rng(0)
control_a = rng.choice(len(vocabulary), len(form_pairs))
control_b = rng.choice(len(vocabulary), len(form_pairs))
morph_same, morph_other = [], []
for seed in SEEDS:
    same = float((unit[seed][form_pairs[:, 0]] * unit[seed][form_pairs[:, 1]]).sum(1).mean())
    other = float((unit[seed][control_a] * unit[seed][control_b]).sum(1).mean())
    morph_same.append(same); morph_other.append(other)
    print(f"зерно {seed}: форми однієї леми {same:.4f} · випадкові пари {other:.4f}"
          f" · розрив {same - other:.4f}")
morph_same = np.array(morph_same); morph_other = np.array(morph_other)
morph_gap = morph_same - morph_other
print()
print(f"у середньому: форми однієї леми {morph_same.mean():.4f} ±{morph_same.std():.4f}"
      f" · випадкові пари {morph_other.mean():.4f} ±{morph_other.std():.4f}")
print(f"розрив у середньому: {morph_gap.mean():.4f} ±{morph_gap.std():.4f}")
print(f"для порівняння, розрив на парах документів однакового змісту: {gap_e.mean():.4f}")

## 10 · Зовнішня оцінка: узяти задачу й подивитись

Внутрішні перевірки міряють простір сам по собі. Зовнішня питає інше: чи стане
краще **справжній задачі**, якщо підкласти їй ембединги.

Беремо задачу теми 06 без жодної зміни. Мітка — з англійського оригіналу, ознаки —
з українського перекладу, тож модель не бачить того, з чого зроблено мітку.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from scipy.sparse import hstack, csr_matrix

ERROR_WORDS = re.compile(
    r'\b(error|failed|cannot|could not|unable|invalid|denied|no such)\b', re.I)
labels = np.array([1 if ERROR_WORDS.search(s) else 0 for s in sources])
print("документів     :", len(documents))
print(f"клас «помилка» : {labels.mean():.4f}")

def document_matrix(matrix):
    """Кожен документ — середнє векторів своїх слів. 64 числа замість
    чотирнадцяти тисяч колонок TF-IDF."""
    out = np.zeros((len(documents), matrix.shape[1]), dtype=np.float32)
    for i, words in enumerate(tokenized):
        ids = [word_index[w] for w in words if w in word_index]
        if ids:
            out[i] = matrix[ids].mean(0)
    return out / np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-9)

dense = {seed: document_matrix(unit[seed]) for seed in SEEDS}
print("розмір щільної матриці ембедингів:", dense[0].shape)

In [ ]:
scores = collections.defaultdict(list)
for seed in SEEDS:
    train_idx, test_idx = train_test_split(np.arange(len(documents)),
                                           test_size=0.3, random_state=seed)
    y_train, y_test = labels[train_idx], labels[test_idx]

    vec = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, min_df=2)
    X_train = vec.fit_transform([documents[i] for i in train_idx])
    X_test = vec.transform([documents[i] for i in test_idx])

    D_train, D_test = dense[seed][train_idx], dense[seed][test_idx]

    plan = [
        ("TF-IDF",                    X_train, X_test, None),
        ("TF-IDF balanced",           X_train, X_test, 'balanced'),
        ("ембединги 64",              D_train, D_test, None),
        ("ембединги 64 balanced",     D_train, D_test, 'balanced'),
        ("TF-IDF + ембединги",        hstack([X_train, csr_matrix(D_train)]).tocsr(),
                                      hstack([X_test, csr_matrix(D_test)]).tocsr(), 'balanced'),
    ]
    for name, A, B, weight in plan:
        model = LogisticRegression(max_iter=1000, class_weight=weight)
        started = time.process_time()
        model.fit(A, y_train)
        spent = time.process_time() - started
        scores[name].append((f1_score(y_test, model.predict(B)), spent))
    print(f"зерно {seed} готово · ознак TF-IDF {X_train.shape[1]}")

print()
print(f"{'схема':<24}{'F1 «помилка»':>17}{'проц. час, с':>15}")
for name, values in scores.items():
    f1 = np.array([a for a, _ in values]); cpu = np.array([b for _, b in values])
    print(f"{name:<24}{f1.mean():>10.4f} ±{f1.std():.4f}{cpu.mean():>15.3f}")

### Головне число теми — і воно не на користь ембедингів

Порахуємо різниці явно й перевіримо кожну проти розкиду.

In [ ]:
def summarize(a, b, label):
    fa = np.array([x for x, _ in scores[a]]); fb = np.array([x for x, _ in scores[b]])
    diff = fb.mean() - fa.mean(); spread = max(fa.std(), fb.std())
    tag = "різниця доведена" if abs(diff) > 2 * spread else "у межах розкиду — не різниця"
    print(f"{label:<44}{diff:+.4f}  (розкид {spread:.4f}) → {tag}")

summarize("TF-IDF", "ембединги 64", "ембединги замість TF-IDF")
summarize("TF-IDF balanced", "ембединги 64 balanced", "те саме зі зважуванням класів")
summarize("TF-IDF balanced", "TF-IDF + ембединги", "ембединги ДОДАНІ до TF-IDF")

### Може, річ у тому, що розмічених прикладів забагато?

Правдоподібне заперечення: ембединги вчаться на **нерозміченому** тексті, тож їхня
перевага мала б проявитись там, де розмічених прикладів мало. Перевіряємо кривою.

In [ ]:
SIZES = [200, 1000, 5000]
curve = collections.defaultdict(lambda: collections.defaultdict(list))

for seed in SEEDS:
    train_idx, test_idx = train_test_split(np.arange(len(documents)),
                                           test_size=0.3, random_state=seed)
    y_test = labels[test_idx]
    order = np.random.default_rng(seed).permutation(train_idx)
    for size in SIZES:
        subset = order[:min(size, len(train_idx))]
        # словник будуємо ЛИШЕ з цих прикладів — інакше TF-IDF крадькома
        # дізнається про весь корпус, і крива бреше на малих розмірах
        vec = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, min_df=2)
        Xs = vec.fit_transform([documents[i] for i in subset])
        Xt = vec.transform([documents[i] for i in test_idx])
        m = LogisticRegression(max_iter=1000, class_weight='balanced').fit(Xs, labels[subset])
        curve[size]['TF-IDF'].append(f1_score(y_test, m.predict(Xt)))
        m = LogisticRegression(max_iter=1000, class_weight='balanced').fit(
            dense[seed][subset], labels[subset])
        curve[size]['ембединги'].append(f1_score(y_test, m.predict(dense[seed][test_idx])))
    print(f"зерно {seed} готово")

print()
# останній рядок — це та сама конфігурація, що вже порахована вище,
# тож беремо її звідти замість того, щоб векторизувати корпус ще тричі
full = len(documents) - int(0.3 * len(documents))
curve[full]['TF-IDF'] = [x for x, _ in scores['TF-IDF balanced']]
curve[full]['ембединги'] = [x for x, _ in scores['ембединги 64 balanced']]

print(f"{'розмічених':>12}{'TF-IDF':>18}{'ембединги 64':>20}")
for size in SIZES + [full]:
    a = np.array(curve[size]['TF-IDF']); b = np.array(curve[size]['ембединги'])
    print(f"{size:>12}{a.mean():>11.4f} ±{a.std():.4f}{b.mean():>13.4f} ±{b.std():.4f}")

## 11 · Упередженість перша: домен

Корпус — переклади інтерфейсів. Класичних гендерних асоціацій у ньому шукати марно:
ані «медсестри», ані «інженера» там просто немає. Зате є інша упередженість, і вона
вимірна: **слово тут означає те, що воно означає в цій програмі**.

Спершу очима — сусіди слів, у яких поза комп'ютером є цілком побутове значення.

In [ ]:
AMBIGUOUS = ['дерево', 'ключ', 'гілка', 'вікно', 'сум', 'поле', 'адреса']
for word in AMBIGUOUS:
    if word not in word_index:
        print(f"{word:10} — у словнику немає (у корпусі {frequency[word]} разів)")
        continue
    print(f"{word:10} ({frequency[word]:5d}) → " + " · ".join(neighbours(0, word, 5)))
print()
for word in ('дерево', 'сум'):
    print(f"— у якому значенні тут «{word}»:")
    shown = 0
    for words, text in zip(tokenized, documents):
        if word in words:
            print("    ", ' '.join(text.split())[:76]); shown += 1
            if shown == 2: break

Тепер число замість враження. У кожного слова є **головна програма** — та, у чиїх
рядках воно трапляється найчастіше. Якщо простір справді розкладено по доменах, то
найближчий сусід слова має походити з тієї самої програми частіше, ніж це вийшло б
випадково.

In [ ]:
word_program = collections.defaultdict(collections.Counter)
for words, program in zip(tokenized, programs):
    for word in set(words):
        if word in word_index:
            word_program[word][program] += 1

program_id = {}
program_name = []
main_program = np.zeros(len(vocabulary), dtype=np.int32)
main_share = np.zeros(len(vocabulary))
for word, i in word_index.items():
    counter = word_program[word]
    best, hits = counter.most_common(1)[0]
    if best not in program_id:
        program_id[best] = len(program_name)
        program_name.append(best)
    main_program[i] = program_id[best]
    main_share[i] = hits / sum(counter.values())

print(f"у середньому на головну програму припадає {main_share.mean():.4f} згадок слова")
print()
for word in AMBIGUOUS:
    if word not in word_index:
        continue
    i = word_index[word]
    print(f"  {word:<9} головна програма {program_name[main_program[i]]:<26}"
          f" {main_share[i]:.4f}")
    for neighbour in neighbours(0, word, 5):
        j = word_index[neighbour]
        mark = "та сама" if main_program[j] == main_program[i] else "інша"
        print(f"        → {neighbour:<18} {program_name[main_program[j]]:<26}"
              f" {mark}")
print()

shares = []
for seed in SEEDS:
    nearest = tops[seed][:, 0]          # перший стовпчик — найближчий сусід
    shares.append(float((main_program[nearest] == main_program).mean()))
rng = np.random.default_rng(0)
chance = float(np.mean(main_program[rng.choice(len(vocabulary), 50000)]
                       == main_program[rng.choice(len(vocabulary), 50000)]))
print(f"найближчий сусід із тієї самої головної програми: "
      f"{np.mean(shares):.4f} ±{np.std(shares):.4f}")
print(f"у випадкової пари слів                          : {chance:.4f}")
print(f"перевищення над випадковістю                    : у {np.mean(shares) / chance:.1f} раза")

## 12 · Упередженість друга: частота

Дистрибутивна семантика каже «слово пізнається за компанією». Але компанія буває
двох різних сортів, і модель їх не розрізняє: слова можуть **стояти поруч**
(«програмним забезпеченням») або **стояти в схожих місцях** («файл» і «каталог»).
Схожими за змістом є другі, а високий косинус дістають обидва.

Порахуємо, скільки разів кожна пара слів словника стояла поруч у вікні ±5.

In [ ]:
from scipy.sparse import coo_matrix

low_id = np.minimum(raw_centers, raw_contexts)
high_id = np.maximum(raw_centers, raw_contexts)
together = coo_matrix((np.ones(len(low_id), dtype=np.float32), (low_id, high_id)),
                      shape=(len(vocabulary), len(vocabulary))).tocsr()
together.sum_duplicates()
print(f"різних пар, що бодай раз стояли поруч: {together.nnz}"
      f" = {100 * together.nnz / (len(vocabulary) ** 2 / 2):.2f} % усіх можливих")

coo = together.tocoo()
pair_i, pair_j, pair_n = coo.row, coo.col, coo.data
# PMI: у скільки разів пара трапляється частіше, ніж якби слова стояли поруч навмання
total_pairs = float(pair_n.sum())
share = word_counts.astype(float) / float(word_counts.sum())
pmi = np.log((pair_n / total_pairs) / (share[pair_i] * share[pair_j]))
cosine_of_pair = (unit[0][pair_i] * unit[0][pair_j]).sum(1)

# однолітерні уламки на кшталт «р» — це залишки скорочень, а не слова;
# у прикладах вони лише заважають, тож у сотню найсильніших їх не пускаємо
long_enough = np.array([len(w) >= 3 for w in vocabulary])
frequent_enough = (pair_n >= 30) & long_enough[pair_i] & long_enough[pair_j]
strongest = np.argsort(-pmi[frequent_enough])[:100]
top_i = pair_i[frequent_enough][strongest]; top_j = pair_j[frequent_enough][strongest]
top_n = pair_n[frequent_enough][strongest]
top_cos = cosine_of_pair[frequent_enough][strongest]
print()
print("вісім найсильніших колокацій корпусу:")
for k in range(8):
    print(f"   {vocabulary[top_i[k]]:>16} · {vocabulary[top_j[k]]:<16}"
          f" поруч {int(top_n[k]):>4} разів"
          f" · косинус {top_cos[k]:.4f}")

Тепер контроль, без якого це нічого не доводить. Слова цих ста колокацій — не
випадкові слова словника: вони мають свою частоту, а частота, як ми щойно бачили,
сама по собі рухає косинус. Тому порівнюємо колокації **не з усім словником, а з
випадковими парами тих самих слів**.

In [ ]:
collocation_words = np.unique(np.r_[top_i, top_j])
print(f"різних слів у сотні найсильніших колокацій: {len(collocation_words)}")

pair_scores, control_scores = [], []
for seed in SEEDS:
    pair_scores.append(float((unit[seed][top_i] * unit[seed][top_j]).sum(1).mean()))
    rng = np.random.default_rng(seed)
    a = rng.choice(collocation_words, 20000); b = rng.choice(collocation_words, 20000)
    different = a != b
    control_scores.append(float((unit[seed][a[different]] * unit[seed][b[different]]).sum(1).mean()))
pair_scores = np.array(pair_scores); control_scores = np.array(control_scores)
print(f"косинус самих колокацій                 : {pair_scores.mean():.4f} ±{pair_scores.std():.4f}")
print(f"косинус випадкових пар З ТИХ САМИХ СЛІВ : {control_scores.mean():.4f} ±{control_scores.std():.4f}")
print(f"надбавка за те, що слова стоять поруч   : {pair_scores.mean() - control_scores.mean():+.4f}")

Те саме, але не на сотні найяскравіших пар, а на всіх — окремо в кожній частотній
смузі, щоб частота не втручалася.

In [ ]:
print(f"{'смуга рангів':>16}{'пар поруч':>12}{'поруч':>10}{'випадкові':>12}{'надбавка':>11}")
for low, high in BANDS:
    band = np.where((rank >= low) & (rank < high))[0]
    inside = np.isin(pair_i, band) & np.isin(pair_j, band) & (pair_n >= 3)
    if inside.sum() < 50:
        continue
    near = float((unit[0][pair_i[inside]] * unit[0][pair_j[inside]]).sum(1).mean())
    base = band_random[(low, high)][0]
    print(f"{f'{low}-{high}':>16}{int(inside.sum()):>12}{near:>10.4f}{base:>12.4f}{near - base:>+11.4f}")

## 13 · Упередженість третя: чого немає взагалі

Найтихіша з трьох. Слова, якого в корпусі немає, для моделі не існує — і вона не
скаржиться, а просто не має що відповісти.

Візьмемо шістдесят найзвичайніших українських слів — сімʼя, їжа, погода, тварини,
почуття, місто — і подивимось, скільки з них модель узагалі знає.

In [ ]:
EVERYDAY = ['мама','тато','брат','сестра','дитина','бабуся','хліб','молоко','сіль','цукор',
 'борщ','вечеря','сніданок','яблуко','картопля','дощ','сніг','сонце','вітер','хмара',
 'дерево','квітка','трава','ліс','річка','море','гора','кіт','собака','кінь','корова',
 'птах','риба','любов','щастя','сум','страх','радість','сміх','сльози',
 'вулиця','місто','село','дорога','школа','лікарня','магазин','базар','церква','поїзд',
 'сорочка','взуття','ліжко','стіл','стілець','вікно','двері','пісня','танець','свято']

TECHNICAL = ['файл', 'помилка', 'каталог', 'сервер', 'пароль', 'сертифікат']

in_corpus = [w for w in EVERYDAY if frequency[w] > 0]
with_vector = [w for w in EVERYDAY if w in word_index]
print(f"побутових слів у списку        : {len(EVERYDAY)}")
print(f"трапляються в корпусі хоч раз  : {len(in_corpus)} "
      f"({100 * len(in_corpus) / len(EVERYDAY):.1f} %)")
print(f"мають вектор (count >= 10)     : {len(with_vector)} "
      f"({100 * len(with_vector) / len(EVERYDAY):.1f} %)")
print()
print("скільки разів кожне побутове слово трапилось у корпусі:")
print("   " + ", ".join(f"{w} {frequency[w]}" for w in EVERYDAY))
print()
print("для порівняння, технічні слова:")
print("   " + ", ".join(f"{w} {frequency[w]}" for w in TECHNICAL))
print()
everyday_total = sum(frequency[w] for w in EVERYDAY)
technical_total = sum(frequency[w] for w in TECHNICAL)
print(f"шістдесят побутових слів разом : {everyday_total} слововживань")
print(f"шість технічних слів разом     : {technical_total} слововживань")
print(f"технічна шістка важча за побутову шістдесятку у "
      f"{technical_total / max(everyday_total, 1):.1f} раза")

### А де ж класична гендерна упередженість?

Про упередженість ембедингів найчастіше розповідають на одному прикладі: у моделі,
навченій на новинах, «лікар» стоїть ближче до «чоловік», а «медсестра» — до «жінка».
Спокуса переказати цей приклад і на нашому корпусі велика, тож перевірмо його чесно.

Щоб такий тест поставити, потрібні дві речі: слова, що позначають стать, і назви
професій. Порахуємо, скільки з них взагалі є в нашому корпусі.

In [ ]:
GENDER_WORDS = ['чоловік', 'жінка', 'хлопець', 'дівчина', 'батько', 'мати',
                'син', 'дочка', 'брат', 'сестра', 'пан', 'пані', 'він', 'вона']
JOB_WORDS = ['інженер', 'лікар', 'медсестра', 'вчитель', 'вчителька', 'програміст',
             'секретар', 'секретарка', 'директор', 'директорка', 'няня', 'водій',
             'науковець', 'керівник', 'керівниця', 'прибиральниця']

print(f"{'слово':<16}{'у корпусі':>11}{'має вектор':>13}")
for group_name, group in (("стать", GENDER_WORDS), ("професія", JOB_WORDS)):
    print(f"— {group_name}:")
    for word in group:
        has = "так" if word in word_index else "ні"
        print(f"   {word:<13}{frequency[word]:>11}{has:>13}")

gender_with_vector = [w for w in GENDER_WORDS if w in word_index]
job_with_vector = [w for w in JOB_WORDS if w in word_index]
print()
print(f"слів про стать із вектором   : {len(gender_with_vector)} із {len(GENDER_WORDS)}"
      f"  {gender_with_vector}")
print(f"назв професій із вектором    : {len(job_with_vector)} із {len(JOB_WORDS)}"
      f"  {job_with_vector}")
print()
if len(gender_with_vector) < 2 or len(job_with_vector) < 2:
    print("Класичний тест на гендерну упередженість на цьому корпусі поставити НЕМОЖЛИВО:")
    print("для нього треба щонайменше по дві пари з кожного боку, а їх немає.")
    print("Це і є результат — не «упередженості немає», а «немає самої теми».")
else:
    print("Пар вистачає, тест можна ставити — дивись домашнє завдання рівня 3.")

І та сама відсутність зсередини самого корпусу: словоформи, які модель відкинула
порогом. Скільки з них можна було б відновити, якби вектор складали із шматків
слова, як це робить FastText із теми 13? Найчесніша оцінка — чи є серед відомих
слів **лема** невідомої форми.

In [ ]:
unknown_forms = [word for word, count in frequency.items() if count < MIN_COUNT]
known = set(vocabulary)
started = time.process_time()
lemma_known = 0
for word in unknown_forms:
    parsed = morph.parse(word)
    if parsed and parsed[0].normal_form in known:
        lemma_known += 1
share_recoverable = lemma_known / len(unknown_forms)
print(f"невідомих словоформ                    : {len(unknown_forms)}")
print(f"їхня лема є серед відомих слів         : {lemma_known}"
      f" ({100 * share_recoverable:.1f} %)")
print(f"розбір зайняв                          : {time.process_time() - started:.1f} с процесорних")
print()
print(f"тобто навіть субслова здатні повернути в модель заледве {100 * share_recoverable:.1f} %"
      f" відкинутих форм;")
print(f"решта {100 * (1 - share_recoverable):.1f} % не має в словнику жодного спорідненого слова,")
print("і жодне складання вектора зі шматків слова їх не врятує.")

## 14 · Що з цього виходить

Зведімо все, що зошит надрукував.

In [ ]:
print(f"ВНУТРІШНЯ ОЦІНКА ({len(pairs)} пар однакового змісту, вибірка теми 05)")
print(f"  сирий косинус : TF-IDF {raw_t.mean():.4f} · ембединги {raw_e.mean():.4f}"
      f"  → «краще у {raw_e.mean() / raw_t.mean():.2f} раза»")
print(f"  розрив        : TF-IDF {gap_t.mean():.4f} · ембединги {gap_e.mean():.4f}"
      f"  → {gap_e.mean() - gap_t.mean():+.4f}")
print(f"  AUC           : TF-IDF {auc_t.mean():.4f} · ембединги {auc_e.mean():.4f}"
      f"  → {auc_e.mean() - auc_t.mean():+.4f}")
print(f"  сума Win + Wout: розрив {gap_sum.mean():.4f} ±{gap_sum.std():.4f}")
print(f"  найкраще налаштування (alpha {best_alpha:.2f}): розрив {sweep[best_alpha][2].mean():.4f}"
      f" ±{sweep[best_alpha][2].std():.4f} · AUC {sweep[best_alpha][3].mean():.4f}")
print()
print("ЗОВНІШНЯ ОЦІНКА (задача теми 06, F1 класу «помилка»)")
for name in ("TF-IDF", "TF-IDF balanced", "ембединги 64 balanced", "TF-IDF + ембединги"):
    f1 = np.array([x for x, _ in scores[name]])
    print(f"  {name:<24}{f1.mean():.4f} ±{f1.std():.4f}")
tfidf_f1 = np.array([x for x, _ in scores["TF-IDF"]])
emb_f1 = np.array([x for x, _ in scores["ембединги 64 balanced"]])
print(f"  рубіж теми 06 — 0.8794 ±0.0013; ембединги дають {emb_f1.mean():.4f}"
      f" ±{emb_f1.std():.4f}, тобто {emb_f1.mean() - tfidf_f1.mean():+.4f}")
print()
print("УПЕРЕДЖЕНІСТЬ")
print(f"  домен      : сусід із тієї самої програми {np.mean(shares):.4f}"
      f" проти {chance:.4f} випадково — у {np.mean(shares) / chance:.1f} раза")
print(f"  сусідство  : колокації {pair_scores.mean():.4f} проти {control_scores.mean():.4f}"
      f" на тих самих словах")
print(f"  гендер     : слів про стать із вектором {len(gender_with_vector)},"
      f" назв професій {len(job_with_vector)} — тест поставити нема на чому")
print(f"  відсутність: {len(with_vector)} побутових слів із {len(EVERYDAY)} мають вектор,"
      f" а шість технічних важать у {technical_total / max(everyday_total, 1):.1f} раза більше")

## 15 · Завдання

### 🟢 Рівень 1

Додай до списку `AMBIGUOUS` пʼять своїх слів, у яких є і побутове, і технічне
значення, і подивись на їхніх сусідів. **Зроблено, якщо** ти назвав хоча б одне
слово, чиї сусіди належать до побутового значення, — або показав числом, що таких
немає.

### 🟡 Рівень 2

Заміни середнє векторів документа на **зважене за IDF** середнє й перезапусти
розділи 6, 7 і 10. **Зроблено, якщо** ти маєш таблицю «сирий косинус / розрив / AUC / F1»
для обох способів і сказав, який із трьох вердиктів від цієї заміни змінився.

### 🔴 Рівень 3

Навчи четверту модель зі `dim=128` і повтори весь розділ 10. **Зроблено, якщо** ти
відповів числом на питання, у чому саме впирається F1 ембедингів: у кількість
вимірів чи в спосіб зводити слова в документ.